
# CRISP-DM Step 4: Modeling — Hands-on Lab (v2)

**วัตถุประสงค์**
- เลือกเทคนิคโมเดลให้ตรงโจทย์ (Classification / Regression / Clustering)
- ฝึก Train/Test + Cross-Validation และประเมินด้วยเมตริกที่เหมาะสม
- ปรับ Hyperparameters (GridSearchCV) เพื่อผลลัพธ์ที่ดีขึ้น
- ทดลองปรับ Threshold และวิเคราะห์ผลกระทบทางธุรกิจ



## 0) สร้างชุดข้อมูลสังเคราะห์ (self-contained)
ฟีเจอร์หลัก: `recency_days, frequency, monetary, avg_orders_per_month, avg_basket_value, product_pref`
เป้าหมาย:
- Classification: `will_buy`
- Regression: `spend_next_month`


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# สุ่มข้อมูลสังเคราะห์
rng = np.random.default_rng(42)
n = 5000

recency_days = rng.integers(0, 200, size=n)
frequency = rng.poisson(5, size=n) + 1
monetary = rng.gamma(shape=2.0, scale=120.0, size=n)
avg_orders_per_month = np.clip(rng.normal(2.5, 1.2, size=n), 0, None)
avg_basket_value = monetary / np.maximum(frequency, 1)

categories = np.array(["Electronics","Grocery","Fashion","Beauty","Home","Sports"])
product_pref = rng.choice(categories, size=n, replace=True)

# Classification target (will_buy)
z = 0.8*(frequency>5) + 0.6*(avg_orders_per_month>2.5) - 0.01*recency_days + 0.001*monetary + rng.normal(0, 0.5, size=n)
prob = 1/(1+np.exp(-z))
will_buy = (rng.random(n) < prob).astype(int)

# Regression target (spend_next_month)
spend_next_month = (0.15*monetary + 3.0*avg_orders_per_month - 0.5*recency_days + rng.normal(0, 20, size=n))
spend_next_month = np.clip(spend_next_month, 0, None)

feat = pd.DataFrame({
    "recency_days": recency_days,
    "frequency": frequency,
    "monetary": monetary.round(2),
    "avg_orders_per_month": avg_orders_per_month.round(2),
    "avg_basket_value": avg_basket_value.round(2),
    "product_pref": product_pref,
    "will_buy": will_buy,
    "spend_next_month": spend_next_month.round(2)
})

display(feat.head())
print("Shape:", feat.shape)



## 1) Classification: Decision Tree + Pipeline + Train/Test


In [ ]:

try:
    from sklearn.model_selection import train_test_split
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay, ConfusionMatrixDisplay
except Exception as e:
    print("กรุณาติดตั้ง scikit-learn: pip install scikit-learn")
    raise e

feature_cols_num = ["recency_days","frequency","monetary","avg_orders_per_month","avg_basket_value"]
feature_cols_cat = ["product_pref"]
X = feat[feature_cols_num + feature_cols_cat].copy()
y = feat["will_buy"].astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), feature_cols_num),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), feature_cols_cat)
])

clf_pipe = Pipeline([
    ("prep", preprocess),
    ("model", DecisionTreeClassifier(random_state=42))
])

clf_pipe.fit(X_tr, y_tr)
y_pred = clf_pipe.predict(X_te)
y_prob = clf_pipe.predict_proba(X_te)[:,1]

print(classification_report(y_te, y_pred, digits=3))
print("ROC-AUC:", roc_auc_score(y_te, y_prob))

RocCurveDisplay.from_predictions(y_te, y_prob)
ConfusionMatrixDisplay.from_predictions(y_te, y_pred)



## 2) Cross-Validation + GridSearchCV (Hyperparameter Tuning)


In [ ]:

from sklearn.model_selection import StratifiedKFold, GridSearchCV

param_grid = {
    "model__max_depth": [3, 5, 7, 9, None],
    "model__min_samples_split": [2, 10, 50],
    "model__min_samples_leaf": [1, 5, 20],
    "model__class_weight": [None, "balanced"]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
gs = GridSearchCV(clf_pipe, param_grid, scoring="roc_auc", cv=cv, n_jobs=-1, verbose=0)
gs.fit(X_tr, y_tr)

print("Best params:", gs.best_params_)
best_model = gs.best_estimator_

y_pred_gs = best_model.predict(X_te)
y_prob_gs = best_model.predict_proba(X_te)[:,1]
print(classification_report(y_te, y_pred_gs, digits=3))
print("ROC-AUC (best):", roc_auc_score(y_te, y_prob_gs))

ConfusionMatrixDisplay.from_predictions(y_te, y_pred_gs)



## 3) Threshold Tuning เพื่อ Optimize ตามธุรกิจ


In [ ]:

import numpy as np
from sklearn.metrics import precision_recall_fscore_support

thresholds = np.linspace(0.05, 0.95, 19)
rows = []
for th in thresholds:
    y_hat = (y_prob_gs >= th).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_te, y_hat, average="binary", zero_division=0)
    rows.append((th, p, r, f1))

th_table = pd.DataFrame(rows, columns=["threshold","precision","recall","f1"]).round(3)
display(th_table)

plt.figure()
plt.plot(th_table["threshold"], th_table["f1"], marker="o")
plt.title("F1 vs Threshold")
plt.xlabel("Threshold")
plt.ylabel("F1")
plt.show()



## 4) Regression: RandomForest Regressor + MAE/RMSE


In [ ]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

target = "spend_next_month"
Xr = feat[feature_cols_num + feature_cols_cat].copy()
yr = feat[target].copy()

from sklearn.model_selection import train_test_split
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.2, random_state=42)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocess_r = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), feature_cols_num),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), feature_cols_cat)
])

rf_reg = Pipeline([
    ("prep", preprocess_r),
    ("model", RandomForestRegressor(n_estimators=300, random_state=42))
])

rf_reg.fit(Xr_tr, yr_tr)
pred = rf_reg.predict(Xr_te)

mae = mean_absolute_error(yr_te, pred)
rmse = mean_squared_error(yr_te, pred, squared=False)
print("MAE:", round(mae, 3), "| RMSE:", round(rmse, 3))

plt.figure()
plt.scatter(yr_te, pred, s=8)
plt.title("Predicted vs Actual (Regression)")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.show()



## 5) Clustering: K-Means + Silhouette + โปรไฟล์คลัสเตอร์


In [ ]:

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

Xc = feat[["recency_days","frequency","monetary"]].copy()
Xc_scaled = StandardScaler().fit_transform(Xc)

km = KMeans(n_clusters=4, n_init=10, random_state=42)
labels = km.fit_predict(Xc_scaled)
sil = silhouette_score(Xc_scaled, labels)
print("Silhouette:", round(sil, 3))

feat_clusters = feat.copy()
feat_clusters["cluster"] = labels

cluster_profile = (feat_clusters
                   .groupby("cluster")[["recency_days","frequency","monetary","avg_orders_per_month","avg_basket_value"]]
                   .mean()
                   .round(2))
display(cluster_profile)



## 6) แบบฝึกหัด (TODO)
1. เปลี่ยนโมเดล Classification เป็น Logistic/RandomForest/Gradient Boosting และเทียบ ROC-AUC, F1
2. ขยาย GridSearch (เพิ่มพารามิเตอร์) แล้วสรุป trade-off ระหว่างคุณภาพกับเวลา
3. ปรับ Threshold แล้วสรุปผลกระทบเชิงธุรกิจ (เลือก threshold ที่เหมาะสม)
4. สร้าง Learning Curve / Validation Curve เพื่อตรวจ over/underfitting
5. Regression: ทดลองแปลงฟีเจอร์ (เช่น log(monetary)) แล้วดูผล MAE/RMSE
6. Clustering: ลองค่า k = 3..8 เทียบ Silhouette แล้วทำอินไซต์แต่ละกลุ่ม
